# Chapter 11 — Fine-Tuning Representation Models for Classification

**Practice Notebook**

This notebook covers four progressively sophisticated ways to fine-tune a BERT-style encoder model:

| Part | Method | Data needed | Key idea |
|---|---|---|---|
| 1 | Supervised fine-tuning (full) | All labeled data | Train BERT + head end-to-end |
| 2 | Layer freezing | All labeled data | Trade compute for speed |
| 3 | Few-shot with SetFit | ~16–32 examples | Contrastive learning |
| 4 | Continued MLM pretraining | Unlabeled domain data | Domain adaptation |
| 5 | Named-Entity Recognition | Token-labeled data | Per-token classification + BIO tags |

**Dataset throughout Parts 1–4:** Rotten Tomatoes (5,331 positive + 5,331 negative movie reviews)  
**Dataset for Part 5:** CoNLL-2003 NER (14,000 annotated sentences)

> Fill in every `# YOUR CODE HERE` block. Run cells in order. The expected outputs are shown as comments.

---
## Part 0 — Theory Warm-Up

Before writing any training code, make sure you understand the concepts. These exercises can be answered with pure Python/NumPy — no transformers needed.

### T1 — F1 Score from Scratch

Throughout this chapter, F1 is the evaluation metric of choice. Accuracy fails on balanced binary datasets when a model is biased toward one class.

**Recall the formulas:**

$$\text{Precision} = \frac{TP}{TP + FP} \quad \text{(of everything I called Positive, how many really were?)}$$

$$\text{Recall} = \frac{TP}{TP + FN} \quad \text{(of all actual Positives, how many did I catch?)}$$

$$F_1 = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

**Task:** Implement `compute_f1` below. Then verify: given the sample predictions, what F1 score do you get? (Expected ≈ 0.727)

In [ ]:
def compute_f1(y_true, y_pred, positive_label=1):
    """
    Compute binary F1 score from scratch.
    
    Args:
        y_true: list of ground-truth labels (0 or 1)
        y_pred: list of predicted labels (0 or 1)
        positive_label: which label is considered 'positive' (default 1)
    Returns:
        f1: float
    """
    # YOUR CODE HERE
    # Hint: count TP, FP, FN by iterating over zip(y_true, y_pred)
    # TP: predicted positive AND actually positive
    # FP: predicted positive BUT actually negative
    # FN: predicted negative BUT actually positive
    pass


# Test it
y_true = [1, 0, 1, 1, 0, 1, 0, 0, 1, 1]
y_pred = [1, 0, 1, 0, 0, 1, 1, 0, 0, 1]

f1 = compute_f1(y_true, y_pred)
print(f"F1 score: {f1:.4f}")  # Expected: ~0.7273

### T2 — BIO Tagging Manual Exercise

NER uses **BIO tags**: **B**eginning, **I**nside, **O**utside.

Rules:
- `B-XXX` = first token of a named entity of type XXX
- `I-XXX` = continuation token of the same entity
- `O` = not part of any entity

Entity types: `PER` (person), `ORG` (organisation), `LOC` (location), `MISC` (miscellaneous)

**Task:** Fill in the correct BIO tags for each sentence below.

```
Sentence 1: "Apple released the iPhone in Cupertino ."
Expected:    B-ORG  O        O   B-MISC O  B-LOC     O

Sentence 2: "Elon Musk founded SpaceX in 2002 ."
Expected:    ????  ????  O       ????   O  O    O

Sentence 3: "The New York Times reported on the United States government ."
Expected:    O   ????  ????  ????  O        O  O   ????    ????   O          O
```

In [ ]:
# Fill in the correct BIO tags for sentences 2 and 3
# Replace each '?' with the correct tag string

sentence2_tokens = ["Elon", "Musk", "founded", "SpaceX", "in", "2002", "."]
sentence2_tags   = ["?",    "?",    "O",       "?",      "O",  "O",    "O"]
# YOUR CODE HERE: fill in the '?' entries

sentence3_tokens = ["The", "New",  "York", "Times", "reported", "on", "the", "United", "States", "government", "."]
sentence3_tags   = ["O",   "?",    "?",    "?",     "O",        "O",  "O",   "?",      "?",      "O",          "O"]
# YOUR CODE HERE: fill in the '?' entries

for token, tag in zip(sentence2_tokens, sentence2_tags):
    print(f"{token:12s} {tag}")
print()
for token, tag in zip(sentence3_tokens, sentence3_tags):
    print(f"{token:12s} {tag}")

### T3 — SetFit Pair Generation Math

SetFit's power comes from manufacturing sentence pairs from a small labeled dataset.

For a class with $n$ sentences, the number of unique within-class (positive) pairs is:

$$\text{positive pairs} = \frac{n(n-1)}{2}$$

**Task:** Write a function that computes the total number of sentence pairs SetFit generates, given the parameters below. Then verify the book's numbers.

In [ ]:
def count_setfit_pairs(n_per_class, n_classes, num_iterations):
    """
    Compute total sentence pairs generated by SetFit.
    
    Args:
        n_per_class: number of labeled examples per class
        n_classes: number of classes
        num_iterations: SetFit num_iterations parameter (pair multiplier)
    Returns:
        total_pairs: int
    """
    # YOUR CODE HERE
    # Step 1: positive pairs per class = n*(n-1)//2
    # Step 2: total samples across all classes
    # Step 3: multiply by num_iterations, then by 2 (pos + neg pairs)
    # Hint: the book's result is 1,280 pairs from 32 examples, num_iterations=20
    pass


total = count_setfit_pairs(n_per_class=16, n_classes=2, num_iterations=20)
print(f"Total pairs generated: {total}")  # Expected: 1280

### T4 — Subtoken Alignment Trace

The hardest part of NER is aligning word-level labels to BERT's subword tokens.

**Rules:**
1. `[CLS]` and `[SEP]` → label **-100** (ignored by loss)
2. First subtoken of a word → use the word's original label
3. Continuation subtokens (same word) → if label was `B-XXX`, change to `I-XXX`; else keep same

**Task:** Manually trace the alignment for the sentence below.

```
Words:      Barack    Obama    visited   Berlin
Word labels: B-PER    I-PER      O       B-LOC

After BERT tokenisation:
Subtokens:  [CLS]  Barack  ##ck  Obama  visited  Ber  ##lin  [SEP]

Aligned labels: ????  ????  ????  ????   ????    ????  ????   ????
```

*(Note: 'Barack' → 'Barack', '##ck' is fictional for this exercise. Real tokenization may differ.)*

In [ ]:
# Fill in the aligned labels for the subtokens
# Use strings: 'B-PER', 'I-PER', 'B-LOC', 'I-LOC', 'O', or -100 (integer for special tokens)

subtokens     = ['[CLS]', 'Barack', '##ck', 'Obama', 'visited', 'Ber', '##lin', '[SEP]']
aligned_labels = [   '?',      '?',    '?',     '?',       '?',   '?',     '?',     '?']
# YOUR CODE HERE: fill in the aligned labels

for tok, lbl in zip(subtokens, aligned_labels):
    print(f"{tok:12s} → {lbl}")

# Expected:
# [CLS]        → -100
# Barack       → B-PER
# ##ck         → I-PER   (continuation of Barack = B-PER → I-PER)
# Obama        → I-PER   (first subtoken, original label is I-PER)
# visited      → O
# Ber          → B-LOC
# ##lin        → I-LOC   (continuation of Berlin = B-LOC → I-LOC)
# [SEP]        → -100

---
## Part 1 — Supervised Classification: Full Fine-Tuning

We fine-tune all 12 encoder blocks of `bert-base-cased` together with a classification head. Both the BERT backbone and the head update jointly — gradients flow from the loss all the way back through every layer.

**Expected result: F1 ≈ 0.85** (vs 0.80 from the frozen approach in Chapter 4)

### 1.1 — Imports and Dataset

In [ ]:
import numpy as np
from datasets import load_dataset, load_metric
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

# Load Rotten Tomatoes dataset
tomatoes = load_dataset("rotten_tomatoes")
train_data, test_data = tomatoes["train"], tomatoes["test"]

print(f"Train size: {len(train_data)}")
print(f"Test  size: {len(test_data)}")
print(f"Sample:     {train_data[0]}")

### 1.2 — Load Model and Tokenizer

`AutoModelForSequenceClassification` loads `bert-base-cased` and automatically attaches a two-class feedforward classification head. Set `num_labels=2` so the head has the right output dimension.

In [ ]:
MODEL_ID = "bert-base-cased"

# YOUR CODE HERE
# Load the model with AutoModelForSequenceClassification (num_labels=2)
# Load the tokenizer with AutoTokenizer
model = ...
tokenizer = ...

### 1.3 — Tokenize and Prepare DataCollator

Use `DataCollatorWithPadding` — it pads each batch dynamically to its longest sequence, not to the global maximum. This saves significant compute on shorter batches.

In [ ]:
# YOUR CODE HERE
# 1. Create a DataCollatorWithPadding using the tokenizer
# 2. Write a preprocess_function that tokenizes examples["text"] with truncation=True
# 3. Apply it to train_data and test_data using .map(batched=True)

data_collator = ...

def preprocess_function(examples):
    # YOUR CODE HERE
    pass

tokenized_train = ...
tokenized_test  = ...

### 1.4 — Evaluation Metric (F1)

Define `compute_metrics` to return the F1 score during evaluation. The `eval_pred` argument is a tuple of `(logits, labels)`. Convert logits to predictions with `np.argmax`.

In [ ]:
# YOUR CODE HERE
# Implement compute_metrics(eval_pred) that:
#   1. Unpacks logits, labels from eval_pred
#   2. Gets predictions via np.argmax(logits, axis=-1)
#   3. Uses load_metric("f1") to compute and return {"f1": ...}

def compute_metrics(eval_pred):
    # YOUR CODE HERE
    pass

### 1.5 — Training Arguments

Key hyperparameters:
- `learning_rate=2e-5` — deliberately small to preserve pretrained weights
- `weight_decay=0.01` — L2 regularisation to prevent overfitting
- `num_train_epochs=1` — one pass is enough when starting from a strong pretrained foundation

In [ ]:
# YOUR CODE HERE
# Create TrainingArguments with:
#   output_dir="model", learning_rate=2e-5, per_device_train_batch_size=16,
#   per_device_eval_batch_size=16, num_train_epochs=1, weight_decay=0.01,
#   save_strategy="epoch", report_to="none"

training_args = ...

### 1.6 — Create Trainer and Train

In [ ]:
# YOUR CODE HERE
# Create a Trainer with: model, training_args, tokenized_train, tokenized_test,
#   tokenizer, data_collator, compute_metrics
# Then call trainer.train()

trainer = ...
trainer.train()

In [ ]:
# YOUR CODE HERE
# Evaluate the trained model
# Expected: eval_f1 ≈ 0.85
results = ...
print(results)

---
## Part 2 — Freezing Layers

Setting `param.requires_grad = False` tells PyTorch to skip computing and storing gradients for that parameter during backprop. The parameter stays in the model but never changes. This trades model quality for training speed.

We will run three experiments:
1. Freeze ALL BERT → train only the classification head
2. Freeze first 10 encoder blocks → train blocks 10–11 + head
3. (Reference) Train everything — already done in Part 1

### 2.1 — Inspect BERT's Parameter Names

In [ ]:
# Reload a fresh model (untrained)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# YOUR CODE HERE
# Print all named parameter names using model.named_parameters()
# You should see: bert.embeddings.*, bert.encoder.layer.0..11.*, bert.pooler.*, classifier.*
for name, param in model.named_parameters():
    print(name)

### 2.2 — Experiment A: Freeze All BERT, Train Only the Head

Freeze everything except parameters whose name starts with `"classifier"`.

In [ ]:
# YOUR CODE HERE
# Loop over model.named_parameters():
#   if name starts with "classifier" → requires_grad = True
#   else → requires_grad = False

for name, param in model.named_parameters():
    # YOUR CODE HERE
    pass

# Confirm: count trainable vs frozen parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"Trainable parameters: {trainable:,}")
print(f"Frozen parameters:    {frozen:,}")

In [ ]:
# YOUR CODE HERE
# Re-use the same training_args, tokenized_train, tokenized_test, data_collator, compute_metrics
# Create a new Trainer with the (now partially frozen) model and call train() then evaluate()
# Expected: eval_f1 ≈ 0.63

trainer_frozen_all = ...
trainer_frozen_all.train()
print(trainer_frozen_all.evaluate())

### 2.3 — Experiment B: Freeze First 10 Encoder Blocks

Encoder block 11 starts at parameter index 165. Freeze everything before that index.

In [ ]:
# Reload fresh model
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)

# YOUR CODE HERE
# Use enumerate(model.named_parameters()) to get (index, name, param)
# Freeze parameters at index < 165, leave the rest trainable

for index, (name, param) in enumerate(model.named_parameters()):
    # YOUR CODE HERE
    pass

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")

In [ ]:
# YOUR CODE HERE
# Train and evaluate the partial-freeze model
# Expected: eval_f1 ≈ 0.80

trainer_partial = ...
trainer_partial.train()
print(trainer_partial.evaluate())

### 2.4 — Reflection

Fill in the table with your results:

| Configuration | F1 Score | Comment |
|---|---|---|
| Ch4 frozen model | 0.80 | No fine-tuning at all |
| Freeze ALL BERT (Part 2.2) | ??? | Only head trains |
| Freeze first 10 blocks (Part 2.3) | ??? | Upper 2 layers + head |
| Full fine-tune (Part 1) | ??? | All 12 blocks + head |

**Question:** Why does freezing all layers perform *worse* than the Chapter 4 frozen approach, even though the Chapter 4 approach also doesn't train BERT? *(Hint: think about what model was used in Chapter 4 vs what model we're using here.)*

---
## Part 3 — Few-Shot Classification with SetFit

SetFit achieves competitive performance with only **~16 examples per class** (32 total) by:
1. Generating sentence pairs (positive = same class, negative = different class)
2. Fine-tuning a `SentenceTransformer` via contrastive learning
3. Training a logistic regression classifier on the specialised embeddings

**Expected result: F1 ≈ 0.85 with only 32 labeled examples**

In [ ]:
# Install setfit if needed: pip install setfit
from setfit import sample_dataset, SetFitModel
from setfit import TrainingArguments as SetFitTrainingArguments
from setfit import Trainer as SetFitTrainer

### 3.1 — Sample Few-Shot Data

Simulate the few-shot setting by sampling only 16 examples per class from the full training set.

In [ ]:
tomatoes = load_dataset("rotten_tomatoes")

# YOUR CODE HERE
# Use sample_dataset(tomatoes["train"], num_samples=16) to get 16 examples per class
sampled_train_data = ...

print(f"Few-shot training size: {len(sampled_train_data)}")  # Expected: 32

### 3.2 — Load a Pretrained SentenceTransformer

We use `sentence-transformers/all-mpnet-base-v2` — one of the best-performing general embedding models on the MTEB benchmark.

In [ ]:
# YOUR CODE HERE
# Load SetFitModel.from_pretrained with "sentence-transformers/all-mpnet-base-v2"
setfit_model = ...

### 3.3 — Configure and Train the SetFit Trainer

- `num_epochs=3` — epochs of contrastive learning on sentence pairs
- `num_iterations=20` — how many pair combinations to generate per sample

In [ ]:
# YOUR CODE HERE
# Create SetFitTrainingArguments with num_epochs=3, num_iterations=20
# Create SetFitTrainer with model, args, train_dataset=sampled_train_data,
#   eval_dataset=tomatoes["test"], metric="f1"
# Call trainer.train()

setfit_args = ...
setfit_trainer = ...
setfit_trainer.train()

In [ ]:
# YOUR CODE HERE
# Evaluate the SetFit model
# Expected: F1 ≈ 0.85 — from only 32 labeled examples!
setfit_results = ...
print(setfit_results)

### 3.4 — Optional: Custom Classification Head

By default, SetFit uses logistic regression. You can replace it with a neural head:

In [ ]:
# YOUR CODE HERE (optional)
# Load SetFitModel with use_differentiable_head=True, head_params={"out_features": 2}
# Re-train and compare F1
pass

---
## Part 4 — Continued Pretraining with Masked Language Modeling

BERT was pretrained on Wikipedia and books. For domain-specific tasks, its vocabulary representations may be weak. By continuing to pretrain on domain data using MLM, we adapt BERT's internal representations before fine-tuning for classification.

**Pipeline:**
```
[General BERT] → [Continued MLM on domain data] → [Fine-tune for classification]
```

MLM trains BERT to predict randomly masked tokens (15% of tokens per sentence). This is unsupervised — we do **not** use the sentiment labels.

In [ ]:
from transformers import AutoModelForMaskedLM, DataCollatorForLanguageModeling, pipeline

### 4.1 — Load the MLM Model

`AutoModelForMaskedLM` attaches a vocabulary-sized prediction head (not a 2-class head). During training it predicts the original masked tokens.

In [ ]:
# YOUR CODE HERE
# Load AutoModelForMaskedLM and AutoTokenizer from "bert-base-cased"
mlm_model = ...
mlm_tokenizer = ...

### 4.2 — Tokenize Without Labels

MLM is unsupervised — we remove the sentiment label column. The DataCollator will create labels automatically by masking tokens.

In [ ]:
tomatoes = load_dataset("rotten_tomatoes")
train_data, test_data = tomatoes["train"], tomatoes["test"]

# YOUR CODE HERE
# 1. Write preprocess_function that tokenizes examples["text"] with truncation=True
# 2. Apply to train_data and test_data with batched=True
# 3. Remove the "label" column from both (tokenized_train.remove_columns("label"))

def preprocess_function(examples):
    # YOUR CODE HERE
    pass

mlm_tokenized_train = ...
mlm_tokenized_test  = ...

### 4.3 — DataCollator for Language Modeling

`DataCollatorForLanguageModeling` randomly masks 15% of tokens at batch-formation time. Each batch gets different masks — the model never sees the same version twice.

In [ ]:
# YOUR CODE HERE
# Create DataCollatorForLanguageModeling with:
#   tokenizer=mlm_tokenizer, mlm=True, mlm_probability=0.15
mlm_data_collator = ...

### 4.4 — Train the MLM Model

We train for 10 epochs (more than classification, because MLM is harder). Save the model so it can be reloaded for classification fine-tuning.

In [ ]:
# YOUR CODE HERE
# Create TrainingArguments with:
#   output_dir="model", learning_rate=2e-5, num_train_epochs=10,
#   per_device_train_batch_size=16, per_device_eval_batch_size=16,
#   weight_decay=0.01, save_strategy="epoch", report_to="none"
mlm_training_args = ...

# Create Trainer with mlm_model, mlm_training_args, mlm_tokenized_train,
#   mlm_tokenized_test, mlm_tokenizer, mlm_data_collator
mlm_trainer = ...

# Save the tokenizer BEFORE training (it won't change)
mlm_tokenizer.save_pretrained("mlm")

mlm_trainer.train()

# Save the domain-adapted model
mlm_model.save_pretrained("mlm")

### 4.5 — Test What the Model Learned

Use the `fill-mask` pipeline to see whether the domain-adapted model produces movie-specific completions.

In [ ]:
# YOUR CODE HERE
# Load a fill-mask pipeline from "bert-base-cased" (base model)
# Run it on: "What a horrible [MASK]!"
# Print the top predictions
base_filler = ...
print("Base BERT predictions:")
for pred in base_filler("What a horrible [MASK]!"):
    print(f"  {pred['sequence']}")

In [ ]:
# YOUR CODE HERE
# Load a fill-mask pipeline from "mlm" (your domain-adapted model)
# Run the same query — predictions should be movie-domain words
# Expected: "movie", "film", "mess", "comedy", "story"
domain_filler = ...
print("Domain-adapted BERT predictions:")
for pred in domain_filler("What a horrible [MASK]!"):
    print(f"  {pred['sequence']}")

### 4.6 — Fine-Tune the Domain-Adapted Model for Classification

Load the saved `"mlm"` model with `AutoModelForSequenceClassification` and fine-tune it for sentiment. The classification results should be at least as good as Part 1, potentially better on domain-specific vocabulary.

In [ ]:
# YOUR CODE HERE
# Load AutoModelForSequenceClassification from the "mlm" directory (num_labels=2)
# Load AutoTokenizer from "mlm"
# Then repeat Part 1 steps: tokenize, Trainer, train, evaluate
pass

---
## Part 5 — Named-Entity Recognition (NER)

NER is **token-level classification** — instead of one label per document, we predict one label per token. We use the CoNLL-2003 dataset with 9 label types (O + 4 entity types × 2 B/I).

The key challenge: BERT's WordPiece tokenizer splits words into subwords (`"Maarten"` → `["Ma", "##arte", "##n"]`), but our labels are at the word level. We must align them.

In [ ]:
import evaluate
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification

### 5.1 — Load and Inspect the CoNLL-2003 Dataset

In [ ]:
# Given — no changes needed here
dataset = load_dataset("conll2003", trust_remote_code=True)

# Inspect one example
example = dataset["train"][848]
print("Tokens:  ", example["tokens"])
print("NER tags:", example["ner_tags"])

# Expected:
# Tokens:   ['Dean', 'Palmer', 'hit', 'his', '30th', 'homer', 'for', 'the', 'Rangers', '.']
# NER tags: [1, 2, 0, 0, 0, 0, 0, 0, 3, 0]

### 5.2 — Build Label Mappings

The 9 NER labels for CoNLL-2003. Notice the pattern: B-tags have odd IDs, I-tags have even IDs. The `align_labels` function exploits this.

In [ ]:
# YOUR CODE HERE
# Build the label2id dict:
#   "O": 0, "B-PER": 1, "I-PER": 2, "B-ORG": 3, "I-ORG": 4,
#   "B-LOC": 5, "I-LOC": 6, "B-MISC": 7, "I-MISC": 8
# Then derive id2label as the reverse mapping

label2id = ...
id2label = ...

print(id2label)

### 5.3 — Explore the Tokenisation Problem

Run the tokenizer on the example tokens and see what happens to multi-subword words.

In [ ]:
# Given — examine what the tokenizer produces
ner_tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

token_ids = ner_tokenizer(
    example["tokens"],
    is_split_into_words=True
)["input_ids"]

sub_tokens = ner_tokenizer.convert_ids_to_tokens(token_ids)
print("Sub-tokens:", sub_tokens)
print("Original tokens:", example["tokens"])
print()
print("Problem: 'homer' split into:", [t for t in sub_tokens if 'home' in t or '##r' == t])
print("We have", len(example['tokens']), "word labels but", len(sub_tokens), "subtokens.")

### 5.4 — Implement `align_labels` ★ Key Exercise ★

This is the most important function in NER fine-tuning. It maps word-level labels to subword-level labels.

**Algorithm:**
```
For each subtoken:
  word_idx = which original word this subtoken came from
  
  if word_idx is None:          → special token ([CLS]/[SEP]) → label -100
  elif first subtoken of word:  → use the word's original label
  else (continuation subtoken): → if label is B-XXX (odd ID) → change to I-XXX (even, +1)
                                  else keep same label
```

In [ ]:
def align_labels(examples):
    """
    Tokenize and align NER labels with subword tokens.
    
    Special tokens ([CLS], [SEP]) get label -100.
    First subtoken of a word gets the word's original label.
    Continuation subtokens get I-XXX (if word was B-XXX) else same label.
    """
    token_ids = ner_tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )
    labels = examples["ner_tags"]
    updated_labels = []

    for index, label in enumerate(labels):
        word_ids = token_ids.word_ids(batch_index=index)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # YOUR CODE HERE
            # Case 1: word_idx is None → special token → append -100
            # Case 2: word_idx != previous_word_idx → first subtoken of new word
            #         → append label[word_idx]
            # Case 3: continuation subtoken
            #         → get label[word_idx]
            #         → if it's a B-tag (label % 2 == 1) → convert to I-tag (label + 1)
            #         → append
            # Update previous_word_idx at end of each iteration
            pass

        updated_labels.append(label_ids)

    token_ids["labels"] = updated_labels
    return token_ids


tokenized = dataset.map(align_labels, batched=True)

# Verify alignment
print("Original NER tags:", dataset["train"][848]["ner_tags"])
print("Aligned labels:   ", tokenized["train"][848]["labels"])
# Expected: [-100, 1, 2, 0, 0, 0, 0, 0, 0, 0, 3, 0, -100]
# (note: -100 for [CLS] and [SEP], and extra -100 for subwords of 'homer'→'home'+'##r')

### 5.5 — Load the Token Classification Model

In [ ]:
# YOUR CODE HERE
# Load AutoModelForTokenClassification from "bert-base-cased" with:
#   num_labels=len(label2id), id2label=id2label, label2id=label2id
ner_model = ...

### 5.6 — DataCollator and Evaluation Metric

`DataCollatorForTokenClassification` pads both token IDs and label sequences together.

`seqeval` evaluates at the **entity level** — `"Dean Palmer"` is one correct prediction only if both tokens are correctly labeled.

In [ ]:
# YOUR CODE HERE
# Create DataCollatorForTokenClassification with ner_tokenizer
ner_data_collator = ...

# Load the seqeval metric
seqeval = evaluate.load("seqeval")

In [ ]:
def compute_ner_metrics(eval_pred):
    """
    Compute entity-level F1 using seqeval.
    Ignores tokens with label -100 (special tokens and padding).
    """
    logits, labels = eval_pred
    # YOUR CODE HERE
    # 1. predictions = np.argmax(logits, axis=2)  ← per-token, so axis=2
    # 2. Build true_predictions and true_labels lists:
    #    for each (prediction, label) pair in zip(predictions, labels):
    #      for each (token_pred, token_label) in zip(prediction, label):
    #        if token_label != -100:
    #          true_predictions.append([id2label[token_pred]])
    #          true_labels.append([id2label[token_label]])
    # 3. results = seqeval.compute(predictions=true_predictions, references=true_labels)
    # 4. return {"f1": results["overall_f1"]}
    pass

### 5.7 — Train the NER Model

In [ ]:
# YOUR CODE HERE
# Create TrainingArguments (same hyperparams as Part 1 but output_dir="ner_model")
# Create Trainer with ner_model, ner_training_args, tokenized["train"], tokenized["test"],
#   ner_tokenizer, ner_data_collator, compute_ner_metrics
# Call trainer.train() then trainer.evaluate()
# Expected: eval_f1 ≈ 0.918

ner_training_args = ...
ner_trainer = ...
ner_trainer.train()
print(ner_trainer.evaluate())

### 5.8 — Save and Run Inference

In [ ]:
# YOUR CODE HERE
# 1. Save the model: ner_trainer.save_model("ner_model")
# 2. Create a token-classification pipeline from "ner_model"
# 3. Run inference on "My name is Maarten."
# Expected: Ma / ##arte / ##n all tagged as B-PER / I-PER / I-PER

ner_trainer.save_model("ner_model")
token_classifier = ...
results = token_classifier("My name is Maarten.")
for r in results:
    print(r)

---
## Chapter 11 Summary

Fill in your measured F1 scores:

| Method | Training data | F1 | Notes |
|---|---|---|---|
| Ch4: frozen pretrained model | 8,500 examples | 0.80 | Zero fine-tuning |
| Part 1: Full fine-tune | 8,500 examples | ??? | All layers + head |
| Part 2A: Freeze all BERT | 8,500 examples | ??? | Head only |
| Part 2B: Freeze first 10 blocks | 8,500 examples | ??? | Top 2 layers + head |
| Part 3: SetFit | **32 examples** | ??? | Contrastive learning |
| Part 4: Continued MLM → fine-tune | 8,500 examples | ??? | Domain adaptation |
| Part 5: NER | CoNLL-2003 | ??? | Token-level, seqeval |

**Key takeaways:**
- Fine-tuning the entire BERT backbone consistently outperforms using frozen representations
- SetFit achieves competitive results with a tiny fraction of the labeled data
- Continued pretraining is most valuable when the target domain differs significantly from Wikipedia/books
- NER requires BIO tagging and careful subtoken label alignment — the `align_labels` function is the heart of the pipeline

**Next:** Chapter 12 applies similar fine-tuning techniques to *generative* (decoder) models — instruction tuning and RLHF.